## 01 — NLP Foundations

#### Purpose

This notebook establishes the Natural Language Processing foundations used throughout the Support Ticket NLP project.

The project will classify customer support tickets into business categories such as:

* Billing
* Technical
* Service
* Account
* General

This notebook focuses on the conceptual NLP pipeline and project architecture before implementing preprocessing, feature engineering, model training, embeddings, neural networks, and transformers in later notebooks.

---

#### 1. Project Objective

Customer support systems receive large volumes of unstructured text such as:

```text
"My internet connection keeps dropping."

"I was charged twice this month."

"I cannot access my account."

"I want to upgrade my service."
```

Unlike structured machine-learning features such as tenure, monthly charges, or contract type, raw text cannot be directly consumed by traditional machine-learning algorithms.

The NLP pipeline therefore converts raw text into numerical representations that models can learn from.

High-level flow:

```text
Support Ticket
      ↓
Text Processing
      ↓
Text Representation
      ↓
Classification Model
      ↓
Predicted Support Category
```

---

#### 2. NLP Project Architecture

The project progressively evaluates multiple text representation and modeling approaches.

```text
Raw Support Ticket Text
          ↓
Text Cleaning
          ↓
Tokenization
          ↓
Vocabulary
          ↓
┌──────────────────────────────────────┐
│        Text Representation           │
│                                      │
│  Bag of Words                        │
│  TF-IDF                              │
│  Embeddings                          │
└──────────────────────────────────────┘
          ↓
┌──────────────────────────────────────┐
│        Modeling Approaches           │
│                                      │
│  Traditional ML                      │
│  Neural Network                      │
│  Transformer                         │
└──────────────────────────────────────┘
          ↓
Model Evaluation
          ↓
MLflow Tracking
          ↓
Model Registration
          ↓
Model Serving
          ↓
Endpoint Testing
```

---

#### 3. Core NLP Terminology

##### Corpus

A **corpus** is the complete collection of text documents used by an NLP system.

For this project:

```text
All support tickets = corpus
```

Example:

```text
Ticket 1: "My internet keeps disconnecting."
Ticket 2: "I was charged twice."
Ticket 3: "I cannot access my account."
```

Together, these tickets form a corpus.

---

##### Document

A **document** is one unit of text within the corpus.

For this project:

```text
One support ticket = one document
```

A document may contain one or multiple sentences.

---

##### Sentence

A document may contain multiple sentences.

Example:

```text
"My internet stopped working.
I restarted the router twice."
```

This document contains two sentences.

---

##### Token

A **token** is a unit of text processed by an NLP system.

For traditional word-based tokenization:

```text
"My internet keeps disconnecting."
```

may become:

```text
["my", "internet", "keeps", "disconnecting"]
```

In later transformer notebooks, tokens may represent whole words, subwords, or other text units.

---

##### Tokenization

**Tokenization** is the process of converting text into tokens.

```text
Raw Text
   ↓
Tokenization
   ↓
Tokens
```

Example:

```text
"I was charged twice"
```

becomes:

```text
["i", "was", "charged", "twice"]
```

---

##### Vocabulary

A **vocabulary** is the collection of unique terms learned from the training corpus.

Example corpus:

```text
"internet connection problem"

"billing payment problem"

"internet service problem"
```

Possible vocabulary:

```text
internet
connection
problem
billing
payment
service
```

Each vocabulary item becomes a potential feature in traditional text representations.

---

#### 4. Why Text Must Be Converted to Numbers

Traditional machine-learning models operate on numerical input.

For structured ML:

```text
tenure             → 12
MonthlyCharges     → 79.50
SeniorCitizen      → 0
```

For NLP, the original input may be:

```text
"My internet keeps disconnecting."
```

The text therefore needs a numerical representation.

Conceptually:

```text
Raw Text
   ↓
Text Representation
   ↓
Numeric Vector
   ↓
Machine-Learning Model
```

Example numerical vector:

```text
[0.00, 0.73, 0.00, 0.42, 0.00, ...]
```

The meaning of those values depends on the representation technique.

---

#### 5. Text Cleaning and Normalization

Traditional NLP pipelines commonly perform some form of text normalization.

Potential operations include:

```text
Lowercasing
Punctuation handling
Whitespace normalization
Stop-word handling
Stemming
Lemmatization
```

These operations should not be applied blindly.

Preprocessing decisions should depend on:

* model type
* business problem
* dataset
* language
* whether semantic information may be lost

Transformer models typically require less manual preprocessing because their pretrained tokenizers expect text that resembles the model's training data.

Detailed implementation is covered in:

```text
02_text_cleaning_and_tokenization
```

---

#### 6. Stop Words

Stop words are common words such as:

```text
the
is
a
an
to
for
of
```

Some traditional NLP pipelines remove them because they may provide limited predictive value.

However, stop-word removal can destroy meaning.

Example:

```text
"internet is working"

"internet is not working"
```

Removing:

```text
not
```

would significantly change the meaning.

Therefore stop-word handling must be treated as a modeling decision rather than an automatic preprocessing requirement.

---

#### 7. Stemming and Lemmatization

Both techniques attempt to reduce word variation.

##### Stemming

Stemming applies rule-based transformations.

Example:

```text
connected
connecting
connection
```

may be reduced toward:

```text
connect
```

The resulting stem does not always need to be a valid dictionary word.

---

##### Lemmatization

Lemmatization attempts to determine a linguistically meaningful base form.

Example:

```text
running
runs
ran
```

may be mapped toward:

```text
run
```

For modern transformer-based models, manual stemming or lemmatization is generally unnecessary unless required by a specific use case.

---

#### 8. Text Representation

Text representation converts language into numerical features.

This project evaluates several approaches.

---

##### 8.1 Bag of Words

Bag of Words represents documents using vocabulary word counts.

Example vocabulary:

```text
internet
billing
problem
refund
service
```

Ticket:

```text
"internet problem internet"
```

could become:

```text
[2, 0, 1, 0, 0]
```

Interpretation:

```text
internet → 2
billing  → 0
problem  → 1
refund   → 0
service  → 0
```

Bag of Words captures word occurrence but does not inherently model semantic meaning or word order.

Implementation:

```text
03_vocabulary_and_bag_of_words
```

---

##### 8.2 TF-IDF

TF-IDF improves on basic word counts by weighting words based on their importance within a document relative to the corpus.

Conceptually:

```text
TF
↓
How frequently does the term occur
in this document?

IDF
↓
How rare or informative is the term
across the corpus?

TF × IDF
↓
TF-IDF score
```

A common term such as:

```text
customer
```

may receive a lower weight.

A more distinctive term such as:

```text
refund
```

may receive a higher weight.

Implementation:

```text
04_tfidf_features
```

---

##### 8.3 Embeddings

Embeddings represent text using dense numerical vectors intended to capture semantic relationships.

Conceptually:

```text
"internet connection problem"
             ↓
Embedding Model
             ↓
[0.18, -0.42, 0.71, ...]
```

Semantically related text can have similar vector representations.

For example:

```text
wifi issue
internet problem
network failure
```

may occupy nearby regions in embedding space.

Embedding foundations are covered in:

```text
07_embeddings_foundations
```

---

#### 9. Document-Term Matrix

Bag of Words and TF-IDF commonly produce a document-term matrix.

Example:

```text
            internet  billing  problem  refund

Ticket 1       1         0        1        0

Ticket 2       0         1        0        1

Ticket 3       1         0        1        0
```

Rows represent:

```text
documents
```

Columns represent:

```text
vocabulary features
```

This follows the standard machine-learning matrix convention:

```text
X.shape =
(number_of_samples, number_of_features)
```

For traditional NLP:

```text
X.shape =
(number_of_documents, vocabulary_size)
```

---

#### 10. Sparse Representations

Traditional Bag-of-Words and TF-IDF representations often contain many zeros.

Example:

```text
[0, 0, 0, 0.72, 0, 0, 0, 0.31, 0, ...]
```

If the vocabulary contains 20,000 terms but a ticket contains only 20 known terms, most feature positions are zero.

Such representations are called:

```text
sparse
```

Libraries such as scikit-learn use sparse matrix structures to efficiently store these features.

---

#### 11. Text Representation vs Classification Model

Text representation and classification are separate stages.

For example:

```text
Raw Ticket
    ↓
TfidfVectorizer
    ↓
TF-IDF Features
    ↓
Logistic Regression
    ↓
Support Category
```

`TfidfVectorizer` is not a classification model.

It performs:

```text
text feature extraction
```

while algorithms such as:

```text
Logistic Regression
Naive Bayes
Linear SVM
```

perform:

```text
classification
```

---

#### 12. Support Ticket Classification

The target problem is multiclass classification.

Example categories:

```text
Billing
Technical
Service
Account
General
```

Example:

```text
"I was charged twice this month."
              ↓
Text Representation
              ↓
Classification Model
              ↓
Billing
```

The learning problem can be represented as:

```text
X = numerical representation of ticket text

y = support ticket category
```

---

#### 13. Traditional NLP Classification Pipeline

The initial production-style baseline will follow:

```text
Support Ticket Dataset
        ↓
Train / Validation / Test Split
        ↓
Text Cleaning
        ↓
TF-IDF Vectorization
        ↓
Traditional ML Model
        ↓
Evaluation
```

Candidate baseline models:

```text
Logistic Regression
Multinomial Naive Bayes
Linear SVM
```

These provide strong reference baselines before introducing more complex approaches.

---

#### 14. Fit vs Transform in NLP

The same training-data discipline used in structured machine learning applies to NLP preprocessing.

For TF-IDF:

```text
Training Data
      ↓
fit()
      ↓
Learn vocabulary
+
Learn IDF statistics
```

Then:

```text
Training Data
Validation Data
Test Data
      ↓
transform()
```

Correct workflow:

```python
X_train_tfidf = vectorizer.fit_transform(X_train)

X_val_tfidf = vectorizer.transform(X_val)

X_test_tfidf = vectorizer.transform(X_test)
```

The vectorizer must not be fitted on validation or test data.

This prevents data leakage.

---

#### 15. NLP Evolution Used in This Project

The project intentionally progresses from simpler representations toward contextual models.

```text
Bag of Words
      ↓
Which words occur?
      ↓
TF-IDF
      ↓
Which words are informative?
      ↓
Embeddings
      ↓
What semantic meaning does the text represent?
      ↓
Neural Networks
      ↓
Can nonlinear patterns be learned from text representations?
      ↓
Transformers
      ↓
What does the text mean in context?
```

Each stage addresses limitations of the previous approach.

---

#### 16. Bag of Words → TF-IDF → Embeddings → Transformers

##### Bag of Words

```text
Strength:
Simple and interpretable

Limitation:
Does not understand meaning or context
```

---

##### TF-IDF

```text
Strength:
Weights informative vocabulary terms

Limitation:
Semantic similarity is not inherently represented
```

---

##### Embeddings

```text
Strength:
Capture semantic relationships

Limitation:
Representation quality depends on the embedding model
and may not fully capture task-specific context
```

---

##### Transformers

```text
Strength:
Produce contextual representations
and model relationships between tokens

Tradeoff:
More computationally complex
```

---

#### 17. Neural Networks in the NLP Pipeline

A neural network can learn nonlinear relationships from numerical text representations.

Example:

```text
Text
  ↓
Embedding / Numeric Representation
  ↓
Neural Network
  ↓
Hidden Representations
  ↓
Support Category
```

The same core neural-network concepts remain applicable:

```text
Weights
Biases
Linear Layers
Activation Functions
Loss
Backpropagation
Optimizer
Epochs
```

Only the input representation changes.

---

#### 18. Transformer Classification

Transformer-based models combine tokenization, learned representations, contextual processing, and classification.

High-level architecture:

```text
Raw Text
   ↓
Transformer Tokenizer
   ↓
Token IDs
   ↓
Token Embeddings
   ↓
Transformer Layers
   ↓
Contextual Representation
   ↓
Classification Head
   ↓
Support Category
```

Detailed transformer architecture is intentionally deferred until:

```text
10_transformer_foundations
```

---

#### 19. Production ML Lifecycle

The final Support Ticket NLP solution will follow an end-to-end ML lifecycle.

```text
Data
 ↓
Preprocessing
 ↓
Feature Engineering
 ↓
Model Training
 ↓
Evaluation
 ↓
Experiment Tracking
 ↓
Best Model Selection
 ↓
Model Registration
 ↓
Model Serving
 ↓
Endpoint Testing
```

The later notebooks cover:

```text
12_mlflow_tracking
13_model_registration
14_model_deployment
15_endpoint_testing
```

---

#### 20. Project Notebook Roadmap

```text
support_ticket_nlp/
│
├── 01_nlp_foundations
├── 02_text_cleaning_and_tokenization
├── 03_vocabulary_and_bag_of_words
├── 04_tfidf_features
├── 05_train_ml_classifier
├── 06_evaluate_nlp_classifier
├── 07_embeddings_foundations
├── 08_embedding_based_classification
├── 09_neural_network_for_text
├── 10_transformer_foundations
├── 11_transformer_ticket_classifier
├── 12_mlflow_tracking
├── 13_model_registration
├── 14_model_deployment
├── 15_endpoint_testing
└── 16_final_comparison_and_conclusion
```

---

#### 21. Key Learnings

1. NLP allows machine-learning systems to work with human language.

2. One support ticket is treated as a document, while the full support-ticket dataset forms a corpus.

3. Tokenization breaks text into smaller processing units called tokens.

4. Vocabulary contains the terms learned from the training corpus.

5. Traditional ML models require text to be converted into numerical features.

6. Bag of Words represents documents using vocabulary occurrence or counts.

7. TF-IDF weights vocabulary terms according to their importance within a document and across the corpus.

8. Bag-of-Words and TF-IDF matrices are commonly sparse.

9. Text representation and classification are separate steps.

10. Embeddings introduce semantic representations.

11. Neural networks can learn nonlinear patterns from numerical text representations.

12. Transformers introduce contextual representations through attention-based architectures.

13. Preprocessing components must be fitted using training data only to prevent data leakage.

14. The project will compare traditional NLP, embeddings, neural networks, and transformer-based approaches before selecting and deploying the final model.

---

#### Conclusion

This notebook established the conceptual architecture for the Support Ticket NLP project.

The core transformation is:

```text
Human Language
      ↓
Text Processing
      ↓
Numerical Representation
      ↓
Machine Learning
      ↓
Support Ticket Classification
```

The project will progressively move through:

```text
Bag of Words
      ↓
TF-IDF
      ↓
Embeddings
      ↓
Neural Networks
      ↓
Transformers
```

before completing the production ML lifecycle through:

```text
MLflow
      ↓
Model Registration
      ↓
Model Serving
      ↓
Endpoint Testing
```

No production model is trained in this notebook.

The purpose of this notebook is to define the concepts and architecture that later implementation notebooks will build upon.

---

#### Next Notebook

```text
02_text_cleaning_and_tokenization
```

The next notebook will:

* load the support-ticket dataset
* validate the required schema
* inspect ticket text and labels
* handle null and empty text
* normalize text safely
* tokenize example tickets
* compare raw and cleaned text
* preserve reusable preprocessing logic
* prepare the dataset for vocabulary and feature-engineering notebooks
